# Session 10: the notebook and the script, together

This is the last notebook that is about *how to work* rather than about the
data. It shows the pairing the whole session is building toward: the script
owns the logic, and the notebook borrows it.

## Borrowing from your own script

`sessions/session-10/solutions/pipeline.py` has the finished pipeline in it.
Here it is, imported rather than copied.

In [ ]:
import os
import sys
from pathlib import Path

here = Path.cwd()
while not (here / "data" / "music.db").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

sys.path.append("sessions/session-10/solutions")

from pipeline import (
    clean,
    load,
    minutes_by_genre,
    minutes_by_month,
)

print("imported four functions, and nothing has run yet")

**Nothing was written and nothing was printed.** The import brought in the
functions without firing off the pipeline, because everything that actually
does the work sits behind:

```python
if __name__ == "__main__":
    main()
```

Read that as "the part that runs when you run this file". Run the file
directly and `main()` happens. Import it and it does not.

So the same file is both a program and a toolbox.

## Now use the pieces

In [ ]:
plays = load()
print(plays.shape)
plays.head(3)

In [ ]:
plays = clean(plays)
plays.dtypes

In [ ]:
minutes_by_genre(plays)

In [ ]:
minutes_by_month(plays).head(6)

Four steps, each one checkable on its own, because each one **takes
something and returns something**. None of them print, and none of them
write files.

That is what makes this possible: if `clean()` printed its results instead
of returning them, there would be nothing to look at here.

## This is also how you debug a script

When the script fails, you do not add print statements. You import the
failing function, call it with real data, and poke at the result.

Try it: the cell below asks for a column that does not exist.

In [ ]:
try:
    plays.groupby("mood")["minutes_played"].sum()
except KeyError as error:
    print("KeyError:", error)
    print("\nthe columns that DO exist:")
    print(list(plays.columns))

## What the pipeline produces

The analysis functions return tables. The output functions write files. They
are separate on purpose: `write_chart()` does not care where the numbers came
from, and `minutes_by_genre()` does not know that files exist.

In [ ]:
summary = minutes_by_genre(plays)
print(summary.to_string(index=False))

print(f"\nElectronic is {summary.loc[0, 'share_pct']}% of all listening")
print(f"the top three genres are "
      f"{summary.head(3)['share_pct'].sum():.0f}% between them")

## Running the whole thing

From a terminal, in the repository root:

```
python3 sessions/session-10/solutions/pipeline.py
```

It prints its progress as it goes:

```
loading from data/music.db ...
  2183 rows
cleaning ...
  2183 rows after cleaning
analysing ...
  8 genres, 24 months
writing output ...
  output/minutes_by_genre.csv
  output/minutes_by_month.png
```

Printing the **row count after each step** costs four lines and tells you
two things: where a crash happened, and whether a step quietly threw away
half your data. That second one is the session 9 lesson arriving somewhere
new.

## The test that matters

Delete `output/` and run it again. If everything comes back, your project is
honest: nothing in it depends on a file you cannot regenerate.

That is your homework, and it catches real mistakes, usually a notebook that
quietly wrote something the script needs.

## When to make the move

| A notebook is right when | Time to make it a script when |
|---|---|
| you are working out the question | you have run the same cells in order three times |
| you want to see each step | you want it to run without you |
| the output is the point | somebody else needs to run it |
| you will read it more than run it | you scroll up to re-run a cell you know works |

It is not either/or. Explore in the notebook, move the settled parts into a
script, and import them back. Nothing gets copied twice, which was exactly
the itch in session 9 when you copied `clean_plays()` from one notebook into
another.